# Does fixture context improve national-team player forecasts?

For a count such as completed tackles, write the player expectation as the average of **team volume × player share** across plausible fixture contexts. A strength rating is a fitted, stat-specific pre-round input. Possession and territory are joint match outcomes. Conditioning on the observed outcomes answers a retrospective question. Forecasting their joint distribution answers the selection-time question.

We test three assumptions separately. Can ratings predict the distribution of national fixture context? Does observed context explain team volume beyond the existing strength model? At a fixed volume, does the allocation among players change with context? A final comparison integrates both effects over the same fixture scenarios. Historical club and international appearances enter the player estimate, with an international-only comparison to test the pooling assumption.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.spatial.distance import pdist
from scipy.stats import binom, poisson
from IPython.display import display

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
from six_nations_fantasy.data.load import DATA_DIR, load_dataset
from six_nations_fantasy.data.fantasy_fixtures import tournament_rounds
from six_nations_fantasy.data.lineup import match_context
from six_nations_fantasy.models.player_share import full_lineups
from six_nations_fantasy.models._share_regression import TeamRate
from six_nations_fantasy.scraping.international import COMPETITIONS

years = [2024, 2025, 2026]
stat = 'completed_tackles'  # Also try 'carries'.
data = load_dataset(DATA_DIR)
teams = pd.read_parquet(DATA_DIR / 'team_performances.parquet')
national = {label for _, label in COMPETITIONS.values()}

## Audit the fixture before defining any predictors

The scraper retains each side's possession bins separately. Their combined mass should be about 2.0. Dividing one side's total by two gives its share of that mass, without forcing each side to sum to one. Territory bins are recorded once and reversed for the away side. Keep the original bin order without assigning unverified time or pitch labels. Exclude the last-ten-minute snapshot. The audit counts missing fixtures too. The tolerance allows half a percentage point of rounding per input bin.

In [ ]:
def bins(prefix):
    cols = sorted(teams.filter(regex=rf'^{prefix}_[0-9]+$'), key=lambda c: int(c.split('_')[-1]))
    if not cols:
        raise ValueError(f'Missing {prefix} columns in team_performances.parquet')
    return cols

poss, terr = bins('poss_percents'), bins('territory_percents')
features = poss + terr
if not teams.home.isin([True, False]).all():
    raise ValueError('Invalid home indicator')
wide = teams.set_index(['match_id', 'home'])[features].unstack().reindex(data.matches.match_id)
home, away = [wide.xs(side, level='home', axis=1).reindex(columns=features) for side in (True, False)]
complete = home.notna().all(axis=1) & away.notna().all(axis=1)
bounded = home.ge(0) & home.le(1) & away.ge(0) & away.le(1)
mass = (home[poss] + away[poss]).sum(axis=1, skipna=False)
mirror = np.isclose(home[terr], away[terr].to_numpy()[:, ::-1]).all(axis=1)
valid = complete & bounded.all(axis=1) & mass.sub(2).abs().le(.01 * len(poss) + 1e-9) & mirror
audit = pd.DataFrame(dict(complete=complete, bounded=bounded.all(axis=1), mirrored=mirror, usable=valid))
display(audit.join(data.matches.set_index('match_id').competition).groupby('competition').agg(['size','sum']))
display(pd.DataFrame({'combined possession': mass, 'home possession mass': home[poss].sum(axis=1, skipna=False)}).describe())
context = teams.loc[teams.match_id.isin(valid.index[valid]), ['match_id','home', *features]]

## Estimate volume and player share

Use the sum of credited player events from complete 23-player sheets as the volume target. A player's baseline is their mean share in the same role, shrunk towards the role mean by five appearances. This treats club and test appearances equally in the pooled estimate. Compare it with test history alone. The context adjustment learns a separate response for each role from the historical fraction of team events credited to that role. Each prediction is normalized over the named 23. This is a per-appearance model. Minutes and substitutions are part of the outcome, and differential exposure could explain an apparent role effect.

For the volume adjustment, regress the log ratio of credited to fitted volume on context. Both adjustments use centered, standardized bins and ridge regularization because the bins are dependent. The adjustments are descriptive associations. A team can change its tactics in response to the score.

In [ ]:
frame = match_context(data.performances, data.matches)
frame['date'] = frame.match_id.map(data.matches.set_index('match_id').date)
players, _, sides = full_lineups(frame, training=True)
if not np.isfinite(players[stat]).all() or (players[stat] < 0).any():
    raise ValueError(f'Expected complete nonnegative {stat}')
keys = ['match_id','home']
group = players.groupby(keys)
players['volume'] = group[stat].transform('sum')
players['fraction'] = players[stat].div(players.volume.where(players.volume > 0))
sides = sides.merge(group[stat].sum().rename('volume').reset_index(), on=keys, validate='1:1')
sides = sides.merge(context, on=keys, how='left', validate='1:1')

def player_weights(history, lineup):
    h = history.dropna(subset=['fraction'])
    role = h.groupby('position_role').fraction.mean()
    form = h.groupby(['player_id','position_role']).fraction.agg(['sum','count'])
    past = lineup[['player_id','position_role']].join(form, on=['player_id','position_role'])
    prior = lineup.position_role.map(role).fillna(1/23).astype(float)
    return ((past['sum'].fillna(0) + 5 * prior) / (past['count'].fillna(0) + 5)).clip(lower=1e-8).to_numpy()

def context_effect(x, response):
    center = x.mean(axis=0)
    scale = x.std(axis=0).clip(lower=.01)
    z = ((x-center)/scale).to_numpy()
    beta = np.linalg.solve(z.T @ z + 10*np.eye(z.shape[1]), z.T @ (response-response.mean(axis=0)))
    return lambda values: ((values-center.to_numpy())/scale.to_numpy()) @ beta

## Walk forward by Six Nations round

Refit the existing team/opposition model on every available earlier complete side. Use earlier internationals with both context rows as potential scenarios. Select up to 20 historical fixtures closest in the four log ratings (team and opponent for each side). Each scenario carries the entire home/away vector. Compare its joint energy score with the unconditional distribution over prior internationals. Lower is better. The ratings assigned to historical donors describe those teams as estimated at the current cutoff.

Evaluate the player allocation with negative log share per credited event. This is the allocation-dependent part of multinomial log loss. Compare count forecasts using player squared error and team absolute error. Recorded match sheets condition every comparison on the same 23 players. This does not evaluate late lineup changes or a full fantasy XV.

In [ ]:
def fixture_matrix(table, columns):
    return table.set_index(keys)[columns].unstack().swaplevel(axis=1).reindex(
        columns=pd.MultiIndex.from_product([[True,False], columns])).dropna()

def energy(samples, actual):
    # Ordered-pair expectation includes the diagonal (distance zero).
    return np.linalg.norm(samples-actual, axis=1).mean() - pdist(samples).sum()/len(samples)**2

scores, context_scores, coverage = [], [], []
for year in years:
    for number, schedule in tournament_rounds(data.matches, year, price_free=True):
        cutoff = schedule.date.min()
        train = sides.loc[sides.date < cutoff].copy()
        history = players.loc[players.date < cutoff]
        known = train.dropna(subset=features)
        tests = sides.loc[sides.match_id.isin(schedule.match_id)]
        donors = train.loc[train.competition.isin(national)]
        pool = fixture_matrix(donors, features)
        targets = fixture_matrix(tests, features)
        coverage.append((year, number, len(schedule), len(targets), len(pool)))
        if train.empty or known.empty or pool.empty or targets.empty:
            continue
        rate = TeamRate(train, train.volume.to_numpy(), tau=.25)
        factors = ['team_multiplier','opponent_multiplier']
        def ratings(table):
            values = table[keys].join(np.log(rate.multipliers(table)[factors]))
            return fixture_matrix(values, factors)
        donor_ratings = ratings(donors).loc[pool.index]
        target_ratings = ratings(tests)
        volume_effect = context_effect(known[features], np.log((known.volume+.5)/(rate.predict(known)+.5)).to_numpy())

        role_counts = history.groupby(keys+['position_role'])[stat].sum().unstack(fill_value=0)
        role_counts = role_counts.reindex(pd.MultiIndex.from_frame(known[keys]))
        roles = role_counts.columns
        log_role_share = np.log((role_counts+.5).div(role_counts.sum(axis=1)+.5*len(roles), axis=0))
        role_effect = context_effect(known[features], log_role_share.to_numpy())
        for match_id, actual in targets.iterrows():
            distances = np.linalg.norm(donor_ratings.to_numpy()-target_ratings.loc[match_id].to_numpy(), axis=1)
            near = pool.iloc[np.argsort(distances, kind='stable')[:20]]
            context_scores.append((year, number, match_id, energy(pool.to_numpy(), actual.to_numpy()),
                                   energy(near.to_numpy(), actual.to_numpy())))
            for side in (True, False):
                row = tests.loc[(tests.match_id == match_id) & (tests.home == side)]
                lineup = players.loc[(players.match_id == match_id) & (players.home == side)]
                count = lineup[stat].to_numpy()
                base_volume = rate.predict(row)[0]
                base = player_weights(history, lineup)
                role_idx = roles.get_indexer(lineup.position_role)
                def evaluate(label, volumes, shares):
                    shares = shares / shares.sum(axis=1, keepdims=True)
                    expected = (np.asarray(volumes).reshape(-1,1) * shares).mean(axis=0)
                    # Poisson volume plus multinomial allocation gives a Poisson player
                    # marginal in each scenario. Known-volume diagnostics use a binomial.
                    means = np.asarray(volumes).reshape(-1,1) * shares
                    if label.startswith('Actual volume'):
                        log_prob = binom.logpmf(count, count.sum(), shares)
                    else:
                        log_prob = poisson.logpmf(count, means)
                    count_nll = -(logsumexp(log_prob, axis=0)-np.log(len(shares))).mean()
                    event_share = expected / expected.sum() if expected.sum() else shares.mean(axis=0)
                    scores.append(dict(year=year, round_number=number, match_id=match_id, home=side, model=label,
                                       volume_AE=abs(expected.sum()-count.sum()), player_MSE=np.mean((expected-count)**2),
                                       count_NLL=count_nll, allocation_loss=-(count*np.log(event_share)).sum(), events=count.sum()))
                evaluate('Ratings + national form', base_volume, player_weights(history[history.competition.isin(national)], lineup)[None,:])
                evaluate('Ratings + pooled form', base_volume, base[None,:])
                for label, scenarios in [('Observed context',actual[side].to_numpy()[None,:]),
                                         ('Forecast context',near[side].to_numpy())]:
                    slopes = role_effect(scenarios)
                    shifts = np.zeros((len(scenarios),len(lineup)))
                    shifts[:,role_idx >= 0] = slopes[:,role_idx[role_idx >= 0]]
                    shares = np.exp(np.log(base) + shifts - logsumexp(np.log(base)+shifts, axis=1, keepdims=True))
                    evaluate(label, base_volume*np.exp(volume_effect(scenarios)), shares)
                    if label == 'Observed context':
                        evaluate('Actual volume + pooled shares',count.sum(),base[None,:])
                        evaluate('Actual volume + context shares',count.sum(),shares)

In [ ]:
display(pd.DataFrame(coverage, columns=['year','round','scheduled','evaluated','prior international fixtures']))
if not scores:
    raise ValueError('No evaluable rounds. Check years, team context coverage and earlier international history.')
joint = pd.DataFrame(context_scores, columns=['year','round','match_id','unconditional','rating neighbours'])
display(joint.groupby('year')[['unconditional','rating neighbours']].mean())
results = pd.DataFrame(scores)
def summary(groups):
    totals = results.groupby(groups).agg(sides=('home','size'), volume_MAE=('volume_AE','mean'),
        player_MSE=('player_MSE','mean'), player_count_NLL=('count_NLL','mean'), loss=('allocation_loss','sum'), events=('events','sum'))
    totals['player_RMSE'] = np.sqrt(totals.pop('player_MSE'))
    totals['loss_per_event'] = totals.pop('loss') / totals.events
    return totals.round(4)
display(summary('model'))
display(summary(['year','model']))

## Reading the comparisons

- Compare national form with pooled form first. A gain supports using club appearances for this stat. It leaves the optimal amount of club-to-test shrinkage open.
- Compare the two actual-volume rows for evidence that match context changes who records the events. A drop in loss per event can occur even if the team volume forecast stays identical.
- Compare observed-context volume MAE with the ratings baseline. Then compare forecast context with that baseline. A gain confined to observed context points to the fixture forecast as a bottleneck.
- Inspect year-to-year consistency and fixture coverage before interpreting small gains. Both sides share a fixture, and several forecasts share a round cutoff. The model predicts conditional means and relative shares. Player count NLL tests a first distributional assumption, Poisson volume and multinomial allocation within each scenario, averaged as a mixture before scoring. Compare within the same information set. Match pace and playing-time variation may produce heavier tails. Inspect calibration before using tail probabilities.